# Debug DSPy Agent with MLflow Tracing

In [5]:
# Install libraries
!pip install -q dspy openai mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 3.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 907.5/907.5 kB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [6]:
# Load API key from Colab Secrets
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

In [9]:
# MLflow traces stored in a local SQLite database
import mlflow

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("dspy_airline_agent")
mlflow.dspy.autolog()

2026/06/07 15:06:50 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/07 15:06:50 INFO mlflow.store.db.utils: Updating database tables
2026/06/07 15:06:53 INFO mlflow.tracking.fluent: Experiment with name 'dspy_airline_agent' does not exist. Creating a new experiment.


In [10]:
# Configure the LM
import dspy

dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"))

### Build an Airline Customer Service Agent with dspy.ReAct

In [11]:
from pydantic import BaseModel

class Date(BaseModel):
  # Somehow LLM is bad at specifying `datetime.datetime`
  year: int
  month: int
  day: int
  hour: int

class UserProfile(BaseModel):
  user_id: str
  name: str
  email: str

class Flight(BaseModel):
  flight_id: str
  date_time: Date
  origin: str
  destination: str
  duration: float
  price: float

class Itinerary(BaseModel):
  confirmation_number: str
  user_profile: UserProfile
  flight: Flight

class Ticket(BaseModel):
  user_request: str
  user_profile: UserProfile

In [12]:
user_database = {
    "Adam": UserProfile(user_id="1", name="Adam", email="adam@gmail.com"),
    "Bob": UserProfile(user_id="2", name="Bob", email="bob@gmail.com"),
    "Chelsie": UserProfile(user_id="3", name="Chelsie", email="chelsie@gmail.com"),
    "David": UserProfile(user_id="4", name="David", email="david@gmail.com"),
}

flight_database = {
    "DA123": Flight(
        flight_id="DA123",
        origin="SFO",
        destination="JFK",
        date_time=Date(year=2025, month=9, day=1, hour=1),
        duration=3,
        price=200,
    ),
    "DA125": Flight(
        flight_id="DA125",
        origin="SFO",
        destination="JFK",
        date_time=Date(year=2025, month=9, day=1, hour=7),
        duration=9,
        price=500,
    ),
    "DA456": Flight(
        flight_id="DA456",
        origin="SFO",
        destination="SNA",
        date_time=Date(year=2025, month=10, day=1, hour=1),
        duration=2,
        price=100,
    ),
    "DA460": Flight(
        flight_id="DA460",
        origin="SFO",
        destination="SNA",
        date_time=Date(year=2025, month=10, day=1, hour=9),
        duration=2,
        price=120,
    ),
}

itinery_database = {}
ticket_database = {}

In [13]:
import random
import string


def fetch_flight_info(date: Date, origin: str, destination: str):
  """Fetch flight information from origin to destination on the given date"""
  flights = []

  for flight_id, flight in flight_database.items():
    if (flight.date_time.year == date.year and flight.date_time.month == date.month and flight.date_time.day == date.day and flight.origin == origin and flight.destination == destination):
      flights.append(flight)
  return flights


def fetch_itinerary(confirmation_number: str):
  """Fetch a booked itinerary information from database"""
  return itinery_database.get(confirmation_number)


def pick_flight(flights: list[Flight]):
  """Pick up the best flight that matches users' request."""
  sorted_flights = sorted(
      flights,
      key=lambda x: (
            x.get("duration") if isinstance(x, dict) else x.duration,
            x.get("price") if isinstance(x, dict) else x.price,
      ),
  )
  return sorted_flights[0]

def generate_id(length=8):
  chars = string.ascii_lowercase + string.digits
  return "".join(random.choices(chars, k=length))


def book_itinerary(flight: Flight, user_profile: UserProfile):
  """Book a flight on behalf of the user."""
  confirmation_number = generate_id()
  while confirmation_number in itinery_database:
    confirmation_number = generate_id()
  itinery_database[confirmation_number] = Itinerary(
      confirmation_number=confirmation_number,
      user_profile=user_profile,
      flight=flight,
  )
  return confirmation_number, itinery_database[confirmation_number]


def cancel_itinerary(confirmation_number: str, user_profile: UserProfile):
  """Cancel an itinerary on behalf of the user."""
  if confirmation_number in itinery_database:
    del itinery_database[confirmation_number]
    return
  raise ValueError("Cannot find the itinerary, please check your confirmation number.")


def get_user_info(name: str):
  """Fetch the user profile from database with given name."""
  return user_database.get(name)


def file_ticket(user_request: str, user_profile: UserProfile):
  """File a customer support ticket if this is something the agent cannot handle."""
  ticket_id = generate_id(length=6)
  ticket_database[ticket_id] = Ticket(
      user_request=user_request,
      user_profile=user_profile,
  )
  return ticket_id

In [14]:
class DSPyAirlineCustomerSerice(dspy.Signature):
  """You are an airline customer service agent. You are given a list of tools to handle user request. You should decide the right tool to use in order to fullfil users' request."""
  user_request: str = dspy.InputField()
  process_result: str = dspy.OutputField(desc="Message that summarizes the process result, and the information users need, e.g., the confirmation_number if it's a flight booking request.")

In [15]:
react = dspy.ReAct(
    DSPyAirlineCustomerSerice,
    tools = [
        fetch_flight_info,
        fetch_itinerary,
        pick_flight,
        book_itinerary,
        cancel_itinerary,
        get_user_info,
        file_ticket,
    ]
)

In [16]:
result = react(user_request="please help me book a flight from SFO to JFK on 09/01/2025, my name is Adam")

In [17]:
result

Prediction(
    trajectory={'thought_0': 'I need to fetch flight information from San Francisco (SFO) to New York (JFK) on September 1st, 2025, in order to find available flights for Adam to book.', 'tool_name_0': 'fetch_flight_info', 'tool_args_0': {'date': {'year': 2025, 'month': 9, 'day': 1, 'hour': 0}, 'origin': 'SFO', 'destination': 'JFK'}, 'observation_0': [Flight(flight_id='DA123', date_time=Date(year=2025, month=9, day=1, hour=1), origin='SFO', destination='JFK', duration=3.0, price=200.0), Flight(flight_id='DA125', date_time=Date(year=2025, month=9, day=1, hour=7), origin='SFO', destination='JFK', duration=9.0, price=500.0)], 'thought_1': 'I have retrieved two available flights from SFO to JFK. Now, I need to pick the best flight for Adam to book. The first flight (DA123) is cheaper and has a shorter duration compared to the second flight (DA125). I will select the first flight (DA123) for booking.', 'tool_name_1': 'pick_flight', 'tool_args_1': {'flights': [{'flight_id': 'DA12

**Note:** MLflow autolog is recording traces in the background. The visual MLflow dashboard is not accessible in Colab as it requires running on a local machine with `mlflow ui`.
The ReAct agent output above shows the full trace (thoughts, tools, observations).

In [24]:
# Print the full trajectory step by step
for key, value in result.trajectory.items():
  print(f"{key}: {value}\n")

thought_0: I need to fetch flight information from San Francisco (SFO) to New York (JFK) on September 1st, 2025, in order to find available flights for Adam to book.

tool_name_0: fetch_flight_info

tool_args_0: {'date': {'year': 2025, 'month': 9, 'day': 1, 'hour': 0}, 'origin': 'SFO', 'destination': 'JFK'}

observation_0: [Flight(flight_id='DA123', date_time=Date(year=2025, month=9, day=1, hour=1), origin='SFO', destination='JFK', duration=3.0, price=200.0), Flight(flight_id='DA125', date_time=Date(year=2025, month=9, day=1, hour=7), origin='SFO', destination='JFK', duration=9.0, price=500.0)]

thought_1: I have retrieved two available flights from SFO to JFK. Now, I need to pick the best flight for Adam to book. The first flight (DA123) is cheaper and has a shorter duration compared to the second flight (DA125). I will select the first flight (DA123) for booking.

tool_name_1: pick_flight

tool_args_1: {'flights': [{'flight_id': 'DA123', 'date_time': {'year': 2025, 'month': 9, 'day':

In [25]:
# Pretty print: thoughts, tools, and observations in order
n_steps = len([k for k in result.trajectory if k.startswith("thought_")])

for i in range(n_steps):
  print(f"=== Step {i} ===")
  print(f"Thought: {result.trajectory.get(f'thought_{i}', '')}")
  print(f"Tool: {result.trajectory.get(f'tool_name_{i}', '')}")
  print(f"Tool Args: {result.trajectory.get(f'tool_args_{i}', '')}")
  print(f"Observation: {result.trajectory.get(f'observation_{i}', '')}")
  print()

print(f"Final answer: {result.process_result}")

=== Step 0 ===
Thought: I need to fetch flight information from San Francisco (SFO) to New York (JFK) on September 1st, 2025, in order to find available flights for Adam to book.
Tool: fetch_flight_info
Tool Args: {'date': {'year': 2025, 'month': 9, 'day': 1, 'hour': 0}, 'origin': 'SFO', 'destination': 'JFK'}
Observation: [Flight(flight_id='DA123', date_time=Date(year=2025, month=9, day=1, hour=1), origin='SFO', destination='JFK', duration=3.0, price=200.0), Flight(flight_id='DA125', date_time=Date(year=2025, month=9, day=1, hour=7), origin='SFO', destination='JFK', duration=9.0, price=500.0)]

=== Step 1 ===
Thought: I have retrieved two available flights from SFO to JFK. Now, I need to pick the best flight for Adam to book. The first flight (DA123) is cheaper and has a shorter duration compared to the second flight (DA125). I will select the first flight (DA123) for booking.
Tool: pick_flight
Tool Args: {'flights': [{'flight_id': 'DA123', 'date_time': {'year': 2025, 'month': 9, 'day'